# Architectural Decision Guide: When to Use MMR vs. Similarity vs. Rerankers (Lecture 48)

In production RAG engineering, there is no single retrieval algorithm suited for every workload. Selecting between standard similarity search, Maximal Marginal Relevance (MMR), and Cross-Encoder Rerankers requires analyzing:
1. **Query Intent**: Factoid needle-in-a-haystack vs. multi-perspective exploratory search.
2. **Corpus Redundancy**: Near-duplicate scrape density vs. curated atomic passages.
3. **Latency Budgets**: Sub-millisecond SLA ($<10$ ms) vs. interactive latency SLA ($<200$ ms).

This notebook implements:
1. Two distinct evaluation benchmarks: Multi-faceted coverage vs. precise factoid retrieval.
2. A programmatic **Retrieval Strategy Decision Engine**.
3. Computational latency benchmarking across retrieval paradigms.

## 1. Setup & Ingestion
Initialize embeddings and multi-domain benchmark corpora.

In [1]:
import os
import time
from typing import List, Dict, Any
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

load_dotenv()
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

benchmark_corpus: List[Document] = [
    # Topic 1: Microservices Advantages
    Document(page_content="Microservice architecture enables independent deployment, continuous delivery, and modular technology choices for each service.", metadata={"domain": "microservices", "aspect": "advantages"}),
    Document(page_content="Independent scalability is a primary advantage of microservices, allowing high-traffic components to scale independently.", metadata={"domain": "microservices", "aspect": "advantages"}),
    # Topic 2: Microservices Disadvantages & Complexity
    Document(page_content="Microservices introduce distributed system complexity, network latency, data consistency challenges, and complex observability requirements.", metadata={"domain": "microservices", "aspect": "disadvantages"}),
    # Topic 3: Exact Factoid Targets
    Document(page_content="The default network listening port for PostgreSQL database server instances is 5432.", metadata={"domain": "factoid", "aspect": "postgres_port"}),
    Document(page_content="The default listening port for MySQL server instances is 3306.", metadata={"domain": "factoid", "aspect": "mysql_port"}),
    Document(page_content="Redis in-memory caching server listens on default port 6379.", metadata={"domain": "factoid", "aspect": "redis_port"}),
]

vector_store = FAISS.from_documents(benchmark_corpus, embeddings)
print(f"Benchmark store created with {vector_store.index.ntotal} documents.")

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_25056\1167864205.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Benchmark store created with 6 documents.


## 2. Benchmark Scenario A: Multi-Faceted Query (MMR Superiority)
Evaluate aspect coverage when querying broad architectural trade-offs.

In [2]:
broad_query = "What are the advantages and architectural challenges of adopting microservices?"

# Similarity Retriever
sim_retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 2})
sim_docs = sim_retriever.invoke(broad_query)

# MMR Retriever
mmr_retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 2, "fetch_k": 4, "lambda_mult": 0.5})
mmr_docs = mmr_retriever.invoke(broad_query)

sim_aspects = set(d.metadata.get("aspect") for d in sim_docs)
mmr_aspects = set(d.metadata.get("aspect") for d in mmr_docs)

print("Scenario A: Broad Multi-Faceted Query:")
print(f"  Similarity Search Aspect Coverage: {sim_aspects}")
print(f"  MMR Search Aspect Coverage:        {mmr_aspects}")

assert len(mmr_aspects) == 2, "MMR must capture both advantages and disadvantages facets"
print("Scenario A verification passed.")

Scenario A: Broad Multi-Faceted Query:
  Similarity Search Aspect Coverage: {'advantages'}
  MMR Search Aspect Coverage:        {'disadvantages', 'advantages'}
Scenario A verification passed.


## 3. Benchmark Scenario B: Precise Factoid Lookup (Similarity / Exact Match Superiority)
Demonstrate why forced diversity can degrade precision in needle-in-a-haystack lookups.

In [3]:
factoid_query = "What is the exact default network port for PostgreSQL server?"

sim_factoid = vector_store.similarity_search(factoid_query, k=1)
# Extreme diversity MMR (lambda=0.1) forces irrelevant documents
mmr_aggressive = vector_store.max_marginal_relevance_search(factoid_query, k=2, fetch_k=4, lambda_mult=0.1)

print("Scenario B: Factoid Needle Lookup:")
print(f"  Standard Similarity Rank 1: {sim_factoid[0].page_content}")
print(f"  Aggressive MMR (lambda=0.1) Rank 2: {mmr_aggressive[1].page_content}")

assert "5432" in sim_factoid[0].page_content, "Similarity search accurately retrieves exact port"
print("Scenario B verification passed.")

Scenario B: Factoid Needle Lookup:
  Standard Similarity Rank 1: The default network listening port for PostgreSQL database server instances is 5432.
  Aggressive MMR (lambda=0.1) Rank 2: Independent scalability is a primary advantage of microservices, allowing high-traffic components to scale independently.
Scenario B verification passed.


## 4. Latency Benchmark across Retrieval Strategies
Measure execution latency across 100 iterations.

In [4]:
iterations = 50

# Benchmark Similarity
start_sim = time.perf_counter()
for _ in range(iterations):
    vector_store.similarity_search(broad_query, k=3)
avg_sim_ms = ((time.perf_counter() - start_sim) / iterations) * 1000

# Benchmark MMR
start_mmr = time.perf_counter()
for _ in range(iterations):
    vector_store.max_marginal_relevance_search(broad_query, k=3, fetch_k=6, lambda_mult=0.5)
avg_mmr_ms = ((time.perf_counter() - start_mmr) / iterations) * 1000

print(f"Average Similarity Search Latency: {avg_sim_ms:.3f} ms")
print(f"Average MMR Search Latency:        {avg_mmr_ms:.3f} ms")
print(f"MMR Overhead Factor:               {avg_mmr_ms / avg_sim_ms:.2f}x")

assert avg_sim_ms < 50.0 and avg_mmr_ms < 50.0, "Both strategies must run in low-millisecond regimes"
print("Latency benchmark assertions passed.")

Average Similarity Search Latency: 5.276 ms
Average MMR Search Latency:        5.348 ms
MMR Overhead Factor:               1.01x
Latency benchmark assertions passed.


## 5. Programmatic Retrieval Strategy Decision Engine
A utility function mapping workload characteristics to optimal retriever configuration.

In [5]:
def select_retrieval_strategy(
    query_type: str,           # 'factoid' | 'exploratory' | 'summarization'
    corpus_redundancy: str,    # 'low' | 'high'
    latency_sla_ms: float      # e.g. 10.0, 100.0
) -> Dict[str, Any]:
    """Determine optimal retrieval strategy based on system constraints."""
    if query_type == "factoid":
        return {
            "strategy": "Hybrid (Dense + BM25) + Reranker",
            "search_type": "similarity",
            "rationale": "Factoid lookup requires exact needle precision; MMR diversity risks introducing non-pertinent distractors."
        }
    elif query_type in ["exploratory", "summarization"] and corpus_redundancy == "high":
        return {
            "strategy": "MMR Retriever",
            "search_type": "mmr",
            "lambda_mult": 0.6,
            "rationale": "High redundancy necessitates diversity penalties to prevent context window saturation."
        }
    elif latency_sla_ms < 5.0:
        return {
            "strategy": "Pure In-Memory Dense Similarity",
            "search_type": "similarity",
            "rationale": "Strict sub-5ms SLA prohibits extra cross-attention or iterative MMR matrix computations."
        }
    else:
        return {
            "strategy": "Hybrid + MMR + Reranker",
            "search_type": "hybrid_mmr",
            "rationale": "Balanced enterprise pipeline: broad hybrid candidate recall, MMR diversity pruning, and final cross-encoder ranking."
        }

plan_1 = select_retrieval_strategy("factoid", "low", 50.0)
plan_2 = select_retrieval_strategy("summarization", "high", 50.0)

print("Factoid Scenario Strategy:     ", plan_1["strategy"])
print("Summarization Scenario Strategy:", plan_2["strategy"])

assert plan_1["search_type"] == "similarity"
assert plan_2["search_type"] == "mmr"
print("Decision engine assertion passed.")

Factoid Scenario Strategy:      Hybrid (Dense + BM25) + Reranker
Summarization Scenario Strategy: MMR Retriever
Decision engine assertion passed.
